# Data Challenge 2 - Partie II : Régression
**Prédiction du nombre d'exacerbations de l'asthme**

**Auteurs :** DURIMEL Yohaldere, MOUSLI Rym, ATHMANE Mohamed Anis

## Contexte et Objectif
L'objectif de ce projet est de développer un modèle de régression pour prédire le nombre d'exacerbations annuelles chez des patients asthmatiques. La variable cible (`post_index_exacerbations365`) correspond à des données de comptage (count data) présentant une forte asymétrie avec une prédominance de zéros. 

L'enjeu principal réside dans l'optimisation d'une métrique stricte : la **Déviance de Poisson**. Notre méthodologie repose sur une ingénierie des caractéristiques orientée métier, l'utilisation de l'algorithme LightGBM (objectif Poisson), et une stratégie avancée de post-traitement des probabilités continues.

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from scipy.optimize import minimize
import optuna 
import warnings
warnings.filterwarnings('ignore')

## 1. Définition de la métrique d'évaluation
La performance du modèle est mesurée selon la formule de la Déviance de Poisson :
$$D = 2 \sum_i \left( y_i \log \frac{y_i}{\hat{y}_i} - (y_i - \hat{y}_i) \right)$$

**Considération mathématique :** La fonction ci-dessous intègre un paramètre de lissage (`epsilon`) afin d'éviter une pénalité infinie (division par zéro) dans le cas où le modèle prédirait exactement $0$ pour un patient ayant cliniquement fait au moins une exacerbation ($y_i > 0$).

In [2]:
# DÉFINITION DE LA MÉTRIQUE 
def poisson_deviance(y_true, y_pred):
    """
    Calcule la déviance de Poisson.
    On ajoute un tout petit epsilon à y_pred pour éviter la division par zéro 
    si le modèle prédit exactement 0 pour un vrai cas > 0.
    """
    eps = 1e-10
    y_pred = np.maximum(y_pred, eps) # sécurité mathématique
    
    # on gère le cas où y_true == 0 car y_true * log(0) pose problème en numpy
    # mais mathématiquement la limite de x*log(x) quand x->0 est 0.
    term1 = np.where(y_true == 0, 0, y_true * np.log(y_true / y_pred))
    term2 = y_true - y_pred
    
    deviance = 2 * np.sum(term1 - term2)
    return deviance

# Moyenne de la métrique pour mieux voir les résultats
def poisson_loglik_score(y_true, mu_pred):
    y_true = np.asarray(y_true, dtype=float)
    mu_pred = np.asarray(mu_pred, dtype=float)
    mu_pred = np.clip(mu_pred, 1e-12, None)
    return np.mean(y_true * np.log(mu_pred) - mu_pred)

## 2. Ingénierie des caractéristiques (Feature Engineering)
Afin d'améliorer la capacité de séparation du modèle sur cette cohorte fortement déséquilibrée, nous créons des variables de second ordre. 
Ces transformations visent à mettre en évidence des profils de gravité clinique à travers divers indicateurs de risque :
- Ratios financiers (ex. coût de traitement par jour d'asthme).
- Termes d'interaction entre l'âge du patient et ses historiques d'hospitalisation/traitement.
- Transformations logarithmiques pour normaliser les distributions des variables monétaires fortement étalées.

In [3]:
def add_features(df):
    df = df.copy()

    if {"total_pre_index_charge", "pre_asthma_days"}.issubset(df.columns):
        df["charge_per_day"] = (df["total_pre_index_charge"] / (df["pre_asthma_days"] + 1)).replace([np.inf, -np.inf], np.nan)

    if {"pre_asthma_pharma_charge", "total_pre_index_charge"}.issubset(df.columns):
        df["pharma_ratio"] = (df["pre_asthma_pharma_charge"] / (df["total_pre_index_charge"] + 1)).replace([np.inf, -np.inf], np.nan)

    if {"total_pre_index_charge", "index_age"}.issubset(df.columns):
        df["charge_age_ratio"] = (df["total_pre_index_charge"] / (df["index_age"] + 1)).replace([np.inf, -np.inf], np.nan)

    if {"index_age", "total_pre_index_charge"}.issubset(df.columns):
        df["age_x_total_charge"] = df["index_age"] * df["total_pre_index_charge"]

    if {"index_age", "pre_asthma_days"}.issubset(df.columns):
        df["age_x_asthma_days"] = df["index_age"] * df["pre_asthma_days"]

    if {"index_age", "adherence"}.issubset(df.columns):
        df["age_x_adherence"] = df["index_age"] * df["adherence"]

    if {"charge_per_day", "pre_asthma_days"}.issubset(df.columns):
        df["charge_per_day_x_days"] = df["charge_per_day"] * df["pre_asthma_days"]

    if {"index_age", "total_pre_index_charge"}.issubset(df.columns):
        df["age_charge_interaction"] = df["index_age"] * df["total_pre_index_charge"]

    if {"pre_asthma_days", "charge_per_day"}.issubset(df.columns):
        df["days_charge_interaction"] = df["pre_asthma_days"] * df["charge_per_day"]

    if "pre_asthma_days" in df.columns:
        df["log1p_pre_asthma_days"] = np.log1p(df["pre_asthma_days"].fillna(0).clip(lower=0))

    if "index_age" in df.columns:
        df["age_squared"] = df["index_age"].fillna(0) ** 2

    if "total_pre_index_charge" in df.columns:
        df["log_total_charge"] = np.log1p(df["total_pre_index_charge"].fillna(0).clip(lower=0))

    for c in df.columns:
        if "charge" in c.lower():
            df[f"log1p_{c}"] = np.log1p(df[c].fillna(0).clip(lower=0))

    return df

# CHARGEMENT ET APPLICATION DES TRANSFORMATIONS
df = pd.read_csv('../data/trainDataDC2.csv.gz')

# on enrichit le dataset avec la fonction ci-dessus
df = add_features(df)

# convertir le texte en "Category" (Évite les erreurs)
for c in df.select_dtypes(include=['object']).columns:
    df[c] = df[c].astype('category')

X = df.drop(["post_index_exacerbations365", "patid", "total_pre_index_charge", "pre_asthma_charge"], axis=1, errors='ignore')
y = df["post_index_exacerbations365"]

# on isole 20% pour le TEST FINAL 
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, shuffle=True, random_state=42)

# n divise les 80% restants en TRAIN et VALIDATION (pour guider la recherche d'hyperparamètres avec Optuna)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, shuffle=True, random_state=42) 

print(f"Taille du Train : {len(X_train)} | Validation : {len(X_val)} | Test : {len(X_test)}")
print(f"Nombre de colonnes utilisées pour apprendre : {X_train.shape[1]}")

Taille du Train : 8745 | Validation : 2915 | Test : 2915
Nombre de colonnes utilisées pour apprendre : 41


## 3. Stratégie de modélisation et Optimisation bayésienne
L'algorithme retenu est **LightGBM**, configuré spécifiquement avec une fonction de perte `poisson`. Cet algorithme gère nativement les relations non-linéaires complexes.

Pour déterminer la topologie optimale des arbres (profondeur, feuilles) et les niveaux de régularisation (L1/L2) tout en évitant le surapprentissage, nous utilisons l'optimisation bayésienne via la bibliothèque **Optuna**. L'entraînement est encadré par une validation croisée stricte (mécanisme de *Early Stopping* sur un jeu de validation dédié).

In [4]:
# RECHERCHE AVEC OPTUNA
def objective(trial):
    # espace de recherche massif pour LightGBM
    param = {
        'objective': 'poisson',
        'metric': 'poisson',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 120),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 300),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'seed': 42
    }

    # préparation des datasets pour LightGBM
    dtrain = lgb.Dataset(X_train, label=y_train)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)

    # entraînement avec Early Stopping si ça ne s'améliore plus
    gbm = lgb.train(
        param,
        dtrain,
        num_boost_round=1500, # max d'arbres
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    # prédiction sur la validation et calcul de la déviance
    preds = gbm.predict(X_val)
    score = poisson_deviance(y_val.values, preds)
    
    return score

print("\nLancement d'Optuna")
# on cherche à minimiser (direction='minimize') la déviance
study = optuna.create_study(direction='minimize')

study.optimize(objective, n_trials=250, show_progress_bar=True)

print("\nMeilleurs hyperparamètres trouvés :")
print(study.best_params)

# AFFICHAGE DES DEUX MÉTRIQUES SUR LE JEU DE VALIDATION
print("\nPERFORMANCES DU MEILLEUR MODÈLE")

# on prépare le dictionnaire avec les paramètres gagnants
best_param_eval = study.best_params.copy()
best_param_eval['objective'] = 'poisson'
best_param_eval['metric'] = 'poisson'
best_param_eval['verbosity'] = -1
best_param_eval['seed'] = 42

# on refait un entraînement rapide juste pour récupérer les prédictions
dtrain_eval = lgb.Dataset(X_train, label=y_train)
dval_eval = lgb.Dataset(X_val, label=y_val, reference=dtrain_eval)

best_gbm = lgb.train(
    best_param_eval,
    dtrain_eval,
    num_boost_round=1500,
    valid_sets=[dval_eval],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

# on prédit sur la validation
preds_val = best_gbm.predict(X_val)

# on calcule les DEUX formules
score_deviance = poisson_deviance(y_val.values, preds_val)
score_loglik = poisson_loglik_score(y_val.values, preds_val)

print(f"-Déviance de Poisson : {score_deviance:.2f}")
print(f"-Mean Log-Likelihood : {score_loglik:.4f}")

[I 2026-03-16 16:44:57,416] A new study created in memory with name: no-name-cc550a73-d9d3-42a5-ba77-b0f308adb105



Lancement d'Optuna


  0%|          | 0/250 [00:00<?, ?it/s]

[I 2026-03-16 16:45:02,397] Trial 0 finished with value: 2207.7371270637104 and parameters: {'learning_rate': 0.0019093509705059544, 'num_leaves': 29, 'max_depth': 9, 'min_data_in_leaf': 249, 'feature_fraction': 0.6949643525669214, 'bagging_fraction': 0.7328795652356532, 'bagging_freq': 5, 'lambda_l1': 4.8475109497642764e-05, 'lambda_l2': 2.229430432212693e-06}. Best is trial 0 with value: 2207.7371270637104.
[I 2026-03-16 16:45:03,580] Trial 1 finished with value: 2218.219221704759 and parameters: {'learning_rate': 0.0012074527904977859, 'num_leaves': 32, 'max_depth': 10, 'min_data_in_leaf': 11, 'feature_fraction': 0.7069351285141465, 'bagging_fraction': 0.5098844958517069, 'bagging_freq': 3, 'lambda_l1': 0.0011998415547679563, 'lambda_l2': 7.2367194147656195e-06}. Best is trial 0 with value: 2207.7371270637104.
[I 2026-03-16 16:45:03,971] Trial 2 finished with value: 2203.7241885122967 and parameters: {'learning_rate': 0.05691483791656806, 'num_leaves': 80, 'max_depth': 7, 'min_data_

## 4. Entraînement définitif et Post-traitement (Optimisation de la calibration)
Le modèle de Gradient Boosting retourne des valeurs d'espérance continues (ex: $0.15$ ou $1.8$). Le format de soumission exigeant des valeurs discrètes (entiers), un arrondissement naïf `round()` tendrait à classer la quasi-totalité des individus à $0$, dégradant sévèrement la déviance pour les vrais positifs.

Pour y pallier, nous appliquons un algorithme de minimisation (`scipy.optimize.minimize`) sur les prédictions du jeu de validation. L'objectif est d'isoler un coefficient scalaire optimal qui permet de décaler la frontière de décision, minimisant ainsi mathématiquement la déviance de Poisson avant la conversion en entiers.

In [5]:
# on redonne un maximum de données au modèle final
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])
dtrain_final = lgb.Dataset(X_train_final, label=y_train_final)

# on entraîne le modèle définitif
final_model = lgb.train(best_param_eval, dtrain_final, num_boost_round=500)

# OPTIMISATION DE L'ARRONDI
# on cherche le multiplicateur parfait sur les données de Validation
val_preds_continuous = final_model.predict(X_val)

def optimize_rounding(preds, true_values):
    def loss_func(coef):
        # on multiplie les probabilités continues par un coefficient avant d'arrondir
        int_preds = np.round(preds * coef[0]).astype(int)
        return poisson_deviance(true_values, int_preds)
    
    # on démarre la recherche autour de 1.0 (l'arrondi normal)
    res = minimize(loss_func, [1.0], method='Nelder-Mead')
    return res.x[0]

best_coef = optimize_rounding(val_preds_continuous, y_val.values)
print(f"Coefficient trouvé : {best_coef:.4f}")

Coefficient trouvé : 4.9313


## 5. Évaluation finale et Génération de la soumission
Le modèle optimisé est mis à l'épreuve sur un jeu de test maintenu en aveugle (20% des données initiales). Le coefficient de calibration est appliqué aux nouvelles prédictions. 

Une correction mathématique est imposée par la nature clinique du problème : le nombre d'exacerbations étant strictement positif ou nul, toute prédiction aberrante (issue du décalage) est mathématiquement bornée à $0$. Le format de sortie est ensuite préparé pour la soumission.

In [6]:
# TEST FINAL
print("\nÉvaluation sur le Test Set (20% des patients jamais vus)")
test_preds_continuous = final_model.predict(X_test)

# arrondi classique  (sans optimisation)
test_preds_baseline = np.round(test_preds_continuous).astype(int)
dev_baseline = poisson_deviance(y_test.values, test_preds_baseline)
ll_baseline = poisson_loglik_score(y_test.values, test_preds_baseline)

# Arrondi optimisé
test_preds_optimized = np.round(test_preds_continuous * best_coef).astype(int)
dev_optimized = poisson_deviance(y_test.values, test_preds_optimized)
ll_optimized = poisson_loglik_score(y_test.values, test_preds_optimized)

print("\nLOG-LIKELIHOOD")
print(f"Score (Arrondi Classique) : {dev_baseline:.2f}")
print(f"Score (Arrondi Optimisé)  : {dev_optimized:.2f}")
print(f"Gain grâce à l'optimisation : {dev_baseline - dev_optimized:.2f}")

print("\nMEAN LOG-LIKELIHOOD")
print(f"Score (Arrondi Classique) : {ll_baseline:.4f}")
print(f"Score (Arrondi Optimisé)  : {ll_optimized:.4f}")


Évaluation sur le Test Set (20% des patients jamais vus)

LOG-LIKELIHOOD
Score (Arrondi Classique) : 26322.78
Score (Arrondi Optimisé)  : 11835.49
Gain grâce à l'optimisation : 14487.29

MEAN LOG-LIKELIHOOD
Score (Arrondi Classique) : -5.4801
Score (Arrondi Optimisé)  : -2.3553


In [ ]:
# on récupère les 'patid' originaux correspondant aux lignes de notre X_test
test_patids = df.loc[X_test.index, 'patid']

predictions = pd.DataFrame({
    'patid': test_patids,
    'prediction': test_preds_optimized  # ce sont tes prédictions avec le coeff optimisé
})

nom_fichier = 'Groupe6_Predictions2_DURIMEL_MOUSLI_ATHMANE.csv'
predictions.to_csv(nom_fichier, index=False)

print("\nDistribution des prédictions dans le fichier final :")
print(predictions['prediction'].value_counts().sort_index())


Distribution des prédictions dans le fichier final :
prediction
0    1082
1    1536
2     242
3      46
4       6
5       3
Name: count, dtype: int64
